# Windows 로컬 eTL RAG 실습 노트북

이 노트북은 `data_etl/` 문서를 사용해서 RAG 파이프라인을 실행하는 Windows 로컬 실습용 노트북입니다.

패키지 설치 셀은 넣지 않았습니다. 이미 프로젝트 실행 환경이 준비되어 있다고 가정합니다.

각 단계는 다음 방식으로 실행합니다.

1. 바로 위 Python 셀에서 `QUERY`, `TOP_K`, `LIMIT` 같은 값을 바꿉니다.
2. 바로 아래 Python 실행 셀을 실행합니다.
3. 결과는 cell output에 출력됩니다.

Windows 환경 차이를 줄이기 위해 `%%bash`를 사용하지 않고, 현재 노트북 커널의 Python으로 스크립트를 실행합니다.


## 1. 프로젝트 폴더 경로 설정

`PROJECT_DIR`만 본인 컴퓨터의 프로젝트 폴더 경로로 고치세요.

Windows 경로 앞에는 `r`을 붙입니다.


In [2]:
from pathlib import Path
import os
import sys

# 예: r"C:\sct-project-graphrag"
PROJECT_DIR = Path(r"D:\snu_python\graphrag")

REQUIRED_MARKERS = ["pyproject.toml", "scripts_etl", "data_etl"]

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"프로젝트 폴더를 찾지 못했습니다: {PROJECT_DIR}")

missing = [marker for marker in REQUIRED_MARKERS if not (PROJECT_DIR / marker).exists()]
if missing:
    raise RuntimeError(
        f"프로젝트 폴더로 보이지 않습니다: {PROJECT_DIR}\n"
        f"없는 항목: {missing}\n"
        "zip이 한 겹 더 들어가서 풀렸는지 확인하고 PROJECT_DIR을 안쪽 폴더로 고치세요."
    )

os.chdir(PROJECT_DIR)
os.environ["PROJECT_DIR"] = str(PROJECT_DIR)

print("프로젝트 폴더:", PROJECT_DIR)
print("현재 작업 폴더:", Path.cwd())
print("노트북 Python:", sys.executable)


프로젝트 폴더: D:\snu_python\graphrag
현재 작업 폴더: d:\snu_python\graphrag
노트북 Python: c:\Users\Dell\miniforge3\envs\openai\python.exe


## 2. API 환경변수 설정

LLM을 호출하는 단계에서 사용합니다.

raw BM25 검색만 실행할 때는 API key가 없어도 됩니다.


In [3]:
import getpass
import os

api_key = getpass.getpass("조별 API key를 입력하세요. raw 검색만 할 경우 Enter를 눌러 건너뛰어도 됩니다: ")

if api_key:
    os.environ["OPENAI_API_KEY"] = api_key

os.environ["BASE_URL"] = "ldi.snu.ac.kr:30000"
os.environ["OPENAI_MODEL"] = "gpt-5.4-mini"
os.environ["OPENAI_REASONING_EFFORT"] = "low"
os.environ["OPENAI_EMBEDDING_MODEL"] = "text-embedding-3-small"

print("환경변수 설정 완료")
print("BASE_URL:", os.environ["BASE_URL"])
print("OPENAI_MODEL:", os.environ["OPENAI_MODEL"])


환경변수 설정 완료
BASE_URL: ldi.snu.ac.kr:30000
OPENAI_MODEL: gpt-5.4-mini


## 3. 실행 helper

Windows 경로, 출력 buffering, 한글 인코딩 문제를 줄이기 위해 모든 스크립트를 아래 helper로 실행합니다.


In [4]:
import subprocess
import sys
import os


def run_script(script_name: str, *args: object) -> None:
    script_path = PROJECT_DIR / script_name
    command = [sys.executable, "-u", str(script_path), *[str(arg) for arg in args]]
    print("$", " ".join(command), flush=True)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"

    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=env,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


print("실행 helper 준비 완료")


실행 helper 준비 완료


## 4. 공통 파일 경로

필요하면 여기에서 `data_etl`을 다른 데이터 폴더로 바꾸면 됩니다.


In [5]:
DATA_DIR = "data_etl"

RAW_PAGES = f"{DATA_DIR}/processed/pdf_pages.jsonl"
TOPICS = f"{DATA_DIR}/processed/guide_topics.jsonl"
TOPIC_INDEX = f"{DATA_DIR}/indexes/topic_embedding_index.npz"
TOPIC_METADATA = f"{DATA_DIR}/indexes/topic_embedding_metadata.json"
TOPIC_FEATURES = f"{DATA_DIR}/processed/topic_features.jsonl"
TOPIC_GRAPH = f"{DATA_DIR}/indexes/topic_graph_with_similarity.json"

print("RAW_PAGES:", RAW_PAGES)
print("TOPICS:", TOPICS)


RAW_PAGES: data_etl/processed/pdf_pages.jsonl
TOPICS: data_etl/processed/guide_topics.jsonl


## 5. Raw text BM25 검색

원문 텍스트에서 바로 키워드 검색을 합니다.

API를 사용하지 않습니다.


In [6]:
QUERY = "과제를 만들고 제출 기간을 설정하려면 어디에서 무엇을 해야 하나?"
TOP_K = 3

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)


QUERY: 과제를 만들고 제출 기간을 설정하려면 어디에서 무엇을 해야 하나?
TOP_K: 3


In [7]:
run_script(
    "scripts_etl/06_bm25_raw_text_demo.py",
    QUERY,
    "--input", RAW_PAGES,
    "--top-k", TOP_K,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\06_bm25_raw_text_demo.py 과제를 만들고 제출 기간을 설정하려면 어디에서 무엇을 해야 하나? --input data_etl/processed/pdf_pages.jsonl --top-k 3
1. score=11.407 file=064_과제_만들기_(과제_전체_옵션_설정_요약)_(📄).md chunk=3
    출제**'옵션을 선택하고, **그룹 세트**를 지정합니다. 기본적으로는 그룹 단위로 성적을 부여하지만, 평가 점수를 학생에게 개별적으로 채점하도록 선택할 경우에는 '**각 학생에게 개별적으로 성적 할당**'을 체크할 수 있습니다. **피어 리뷰** ![Untitled]() 학생들이 서로의 과제를 검토하고 피드백 하도록 피어 리뷰 과제를 생성하려면 '**피어 리뷰 진행**' 항목을 체크합니다
2. score=11.150 file=054_학습_활동_-_과제_추가하기_(📄).md chunk=2
   있습니다. 따라서 제출해야 하는지 여부에 대한 혼동을 피하기 위해 과제에 대한 설명에 온라인 제출이 필요한 과제인지 아닌 지를 설명하는 것이 가장 좋습니다. **제출 허용 횟수** ![Untitled]() 온라인 제출 유형을 선택한 경우 과제에 대한 제출 시도를 제한 할 수 있습니다 . 기본적으로 제출 횟수는 **제한 없음[1]**으로 설정됩니다. 이 옵션을 선택하면 학생은 필요한 만큼 과제
3. score=10.491 file=054_학습_활동_-_과제_추가하기_(📄).md chunk=3
   을 설정합니다. 학습 인정 기간은 주차의 학습 인정 기간을 따라 자동 입력됩니다. 필요에 따라 세부 학습 활동의 학습 인정 기간을 변경할 수 있습니다. - **열람 시작일시:** 과제를 제출할 수 있는 시작 시점을 의미합니다. - **마감일시:** 정해진 출석 기간을 마감하는 마감 시점을 의미합니다. 학

## 6. Raw text BM25 RAG 답변

원문 검색 결과를 LLM에게 넣어 답변을 생성합니다.

API를 사용합니다.


In [8]:
QUERY = "동영상 출결을 확인하려면 어디에서 어떤 정보를 봐야 하나?"
TOP_K = 5
MAX_CONTEXT_CHARS = 7000

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)


QUERY: 동영상 출결을 확인하려면 어디에서 어떤 정보를 봐야 하나?
TOP_K: 5


In [9]:
run_script(
    "scripts_etl/09_bm25_raw_rag_answer.py",
    QUERY,
    "--input", RAW_PAGES,
    "--top-k", TOP_K,
    "--max-context-chars", MAX_CONTEXT_CHARS,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\09_bm25_raw_rag_answer.py 동영상 출결을 확인하려면 어디에서 어떤 정보를 봐야 하나? --input data_etl/processed/pdf_pages.jsonl --top-k 5 --max-context-chars 7000
## Retrieved Chunks
1. score=4.733 file=014_Q._복습_영상을_업로드하고_싶습니다.md chunk=0
   [page 1] # Q. 복습 영상을 업로드하고 싶습니다.  - notion_page_id: `fc4bad4e-2a25-4e34-9e13-58965a03d77c`  **Q. 대면 수업에 출석한 학생들을 위한 복습용 영상을 업로드하고 싶습니다. (강의 재생 시 학습자가 재생 구간을 자유롭게 이동하도록 설정하고 싶습니다.)** > **하기 방법은 ****출결 대상이 아닌 ‘복습 영상 업로드 방법
2. score=4.639 file=059_학습_요소_-_MyCMS에서_동영상_및_강의_자료_업로드하기.md chunk=0
   [page 1] # 학습 요소 -  MyCMS에서 동영상 및 강의 자료 업로드하기   - notion_page_id: `af184545-b1d1-4480-b780-93a1fb5de088`  - **동영상 가이드** > **MyCMS 기능은... ☞  CMS 또는 PC에 저장된 다양한 유형의 강의 자료를 업로드하는 기능입니다. ** **☞  동영상 자료 뿐 아니라 각종 유형(PDF, PPT, 
3. score=4.508 file=058_학습_요소_-_MyPC에서_동영상_강의_자료_업로드하기.md chunk=0
   [page 1] # 학습 요소 - MyPC에서 동영상 강의 자료 업로드하기   - notion_page_id: `90b812df-9b24-4731-b043-d93390c042d9`  > **MyPC 기능

## 7. eTL topic 구조화 추출

원문 문서를 읽고, 사용자가 물어볼 만한 질문과 절차를 구조화합니다.

API를 사용합니다.

처음 테스트할 때는 `LIMIT`을 작게 두는 것이 좋습니다.


In [10]:
LIMIT = 5
CONCURRENCY = 2

print("LIMIT:", LIMIT)
print("CONCURRENCY:", CONCURRENCY)
print("OUTPUT:", TOPICS)


LIMIT: 5
CONCURRENCY: 2
OUTPUT: data_etl/processed/guide_topics.jsonl


In [11]:
run_script(
    "scripts_etl/04_extract_guide_topics.py",
    RAW_PAGES,
    TOPICS,
    "--limit", LIMIT,
    "--concurrency", CONCURRENCY,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\04_extract_guide_topics.py data_etl/processed/pdf_pages.jsonl data_etl/processed/guide_topics.jsonl --limit 5 --concurrency 2
[info] 입력에서 고유 문서 5건 추출
[info] 총 5건 처리 시작 (model=gpt-5.4-mini, reasoning_effort=low, concurrency=2, num_ctx=16384)
[ok]   (1/5) 001_FAQ_모음.md — 공통 / 공통 / topic 1개
[ok]   (2/5) 000_📑서울대학교_new_eTL_상세_가이드_(교수자용).md — 교수자, 조교 / 공통 / topic 5개
[ok]   (3/5) 002_과제.md — 교수자, 조교 / 과제 / topic 3개
[ok]   (4/5) 004_Q._과제_제출물_일괄_다운로드가_불가합니다.md — 교수자, 조교 / 과제 / topic 1개
[ok]   (5/5) 003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md — 교수자, 조교 / 과제, 시험, 성적 / topic 2개
[info] 완료 → data_etl\processed\guide_topics.jsonl


## 8. Topic BM25 검색

04번에서 구조화한 topic을 검색합니다.

API를 사용하지 않습니다.


In [12]:
QUERY = "과제 제출 기간과 파일 제출 설정"
TOP_K = 5

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)


QUERY: 과제 제출 기간과 파일 제출 설정
TOP_K: 5


In [13]:
run_script(
    "scripts_etl/07_bm25_topic_demo.py",
    QUERY,
    "--input", TOPICS,
    "--top-k", TOP_K,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\07_bm25_topic_demo.py 과제 제출 기간과 파일 제출 설정 --input data_etl/processed/guide_topics.jsonl --top-k 5
1. score=6.246 file=004_Q._과제_제출물_일괄_다운로드가_불가합니다.md topic=0
   질문: 과제 제출물 일괄 다운로드가 중간에 멈출 때 어떻게 해결하나요?
   결과: 문제가 되는 파일을 제거하거나 제출 형식을 조정한 뒤 과제 제출물을 정상적으로 확인하거나 다운로드할 수 있다.
2. score=4.044 file=003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md topic=0
   질문: 오프라인으로 채점한 과제나 시험 점수를 new eTL에 입력하려면 어떻게 해야 하나요?
   결과: 오프라인 과제 또는 시험용 과제가 생성되고, 성적 메뉴에서 학습자별 점수와 코멘트를 입력할 수 있다.
3. score=3.792 file=002_과제.md topic=1
   질문: 과제 제출물을 한 번에 다운로드할 수 없을 때 어떻게 하나요?
   결과: 과제 제출물 다운로드 가능 여부와 대안 절차를 확인할 수 있다.
4. score=3.127 file=003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md topic=1
   질문: CSV 파일로 과제/시험 성적을 일괄 등록하려면 어떻게 하나요?
   결과: CSV 파일을 이용해 여러 학생의 성적을 한 번에 등록하거나 수정한 내용이 성적 메뉴에 반영된다.
5. score=3.095 file=000_📑서울대학교_new_eTL_상세_가이드_(교수자용).md topic=4
   질문: 과제, 퀴즈, 토론, 파일, 화상강의, 출결 같은 학습 활동은 어떻게 추가하나요?
   결과: 모듈 또는 주차학습 안에 원하는 학습 활동과 요소

## 9. Topic embedding index 생성

구조화된 topic을 embedding index로 만듭니다.

API를 사용합니다.


In [14]:
LIMIT = 50
BATCH_SIZE = 64

print("LIMIT:", LIMIT)
print("BATCH_SIZE:", BATCH_SIZE)
print("INDEX:", TOPIC_INDEX)


LIMIT: 50
BATCH_SIZE: 64
INDEX: data_etl/indexes/topic_embedding_index.npz


In [15]:
run_script(
    "scripts_etl/05_build_topic_embedding_index.py",
    "--input", TOPICS,
    "--index", TOPIC_INDEX,
    "--metadata", TOPIC_METADATA,
    "--limit", LIMIT,
    "--batch-size", BATCH_SIZE,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\05_build_topic_embedding_index.py --input data_etl/processed/guide_topics.jsonl --index data_etl/indexes/topic_embedding_index.npz --metadata data_etl/indexes/topic_embedding_metadata.json --limit 50 --batch-size 64
topics=12 dim=1536
data_etl\indexes\topic_embedding_index.npz
data_etl\indexes\topic_embedding_metadata.json


## 10. Dense topic 검색

질문을 embedding으로 바꾸고, topic embedding index에서 의미 검색을 합니다.

API를 사용합니다.


In [16]:
QUERY = "학생들이 제출한 과제를 확인하고 성적을 처리하는 방법"
TOP_K = 5

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)


QUERY: 학생들이 제출한 과제를 확인하고 성적을 처리하는 방법
TOP_K: 5


In [17]:
run_script(
    "scripts_etl/08_dense_topic_search.py",
    QUERY,
    "--index", TOPIC_INDEX,
    "--metadata", TOPIC_METADATA,
    "--top-k", TOP_K,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\08_dense_topic_search.py 학생들이 제출한 과제를 확인하고 성적을 처리하는 방법 --index data_etl/indexes/topic_embedding_index.npz --metadata data_etl/indexes/topic_embedding_metadata.json --top-k 5
1. score=0.461 file=003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md topic=1
   작업/질문: CSV 파일로 과제/시험 성적을 일괄 등록하려면 어떻게 하나요?
   결과: CSV 파일을 이용해 여러 학생의 성적을 한 번에 등록하거나 수정한 내용이 성적 메뉴에 반영된다.
2. score=0.442 file=002_과제.md topic=2
   작업/질문: 피어리뷰 과제에서 학생별 완료 여부와 미완료 여부를 어떻게 확인하나요?
   결과: 피어리뷰 과제에 대해 학생별 완료 및 미완료 상태를 확인할 수 있다.
3. score=0.408 file=004_Q._과제_제출물_일괄_다운로드가_불가합니다.md topic=0
   작업/질문: 과제 제출물 일괄 다운로드가 중간에 멈출 때 어떻게 해결하나요?
   결과: 문제가 되는 파일을 제거하거나 제출 형식을 조정한 뒤 과제 제출물을 정상적으로 확인하거나 다운로드할 수 있다.
4. score=0.396 file=002_과제.md topic=0
   작업/질문: 오프라인 과제나 시험 점수를 new eTL에 입력하려면 어떻게 하나요?
   결과: 학생별 오프라인 과제 또는 시험 점수가 new eTL에 반영된다.
5. score=0.385 file=003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md topic=0
   작업/질문: 오프라인으로 채점한 과제나 시험 점수를 new eTL에 입력하려

## 11. Topic BM25 + raw text RAG 답변

구조화 topic 검색 결과와 원문 context를 함께 사용해 답변합니다.

API를 사용합니다.


In [18]:
QUERY = "과제를 만들고 제출 현황을 확인한 다음 성적 처리는 어디에서 해야 하나?"
TOP_K = 4
RAW_CONTEXT = "same-document"

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)
print("RAW_CONTEXT:", RAW_CONTEXT)


QUERY: 과제를 만들고 제출 현황을 확인한 다음 성적 처리는 어디에서 해야 하나?
TOP_K: 4
RAW_CONTEXT: same-document


In [19]:
run_script(
    "scripts_etl/10_bm25_topic_rag_answer.py",
    QUERY,
    "--topics", TOPICS,
    "--pages", RAW_PAGES,
    "--top-k", TOP_K,
    "--raw-context", RAW_CONTEXT,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\10_bm25_topic_rag_answer.py 과제를 만들고 제출 현황을 확인한 다음 성적 처리는 어디에서 해야 하나? --topics data_etl/processed/guide_topics.jsonl --pages data_etl/processed/pdf_pages.jsonl --top-k 4 --raw-context same-document
## Retrieved Topics
1. score=7.793 file=003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md topic=0
   질문: 오프라인으로 채점한 과제나 시험 점수를 new eTL에 입력하려면 어떻게 해야 하나요?
   결과: 오프라인 과제 또는 시험용 과제가 생성되고, 성적 메뉴에서 학습자별 점수와 코멘트를 입력할 수 있다.
2. score=2.825 file=003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md topic=1
   질문: CSV 파일로 과제/시험 성적을 일괄 등록하려면 어떻게 하나요?
   결과: CSV 파일을 이용해 여러 학생의 성적을 한 번에 등록하거나 수정한 내용이 성적 메뉴에 반영된다.
3. score=2.475 file=000_📑서울대학교_new_eTL_상세_가이드_(교수자용).md topic=1
   질문: 조교나 청강생을 수동으로 등록하려면 어디에서 처리하나요?
   결과: 선택한 사용자가 해당 과목에 조교 또는 청강생으로 등록된다.
4. score=2.454 file=000_📑서울대학교_new_eTL_상세_가이드_(교수자용).md topic=3
   질문: 게시판을 만들고 게시글을 작성하거나 관리하려면 어떻게 하나요?
   결과: 과목 내에 게시판이 생성되고 게시글을 운영할 수 있다.

## Added Raw Chunks
R1. score=0.000 f

## 12. Search agent demo

agent가 필요한 검색어를 스스로 고르고, 여러 번 검색한 뒤 답변합니다.

API를 사용합니다.


In [20]:
QUERY = "과제를 새로 만들고 제출 기간과 파일 제출을 설정한 뒤, 학생 제출 현황이나 성적 처리는 어디에서 확인해야 하나?"
TOP_K = 4
MAX_STEPS = 4

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)
print("MAX_STEPS:", MAX_STEPS)


QUERY: 과제를 새로 만들고 제출 기간과 파일 제출을 설정한 뒤, 학생 제출 현황이나 성적 처리는 어디에서 확인해야 하나?
TOP_K: 4
MAX_STEPS: 4


In [21]:
run_script(
    "scripts_etl/15_search_agent_demo.py",
    QUERY,
    "--topics", TOPICS,
    "--pages", RAW_PAGES,
    "--top-k", TOP_K,
    "--max-steps", MAX_STEPS,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\15_search_agent_demo.py 과제를 새로 만들고 제출 기간과 파일 제출을 설정한 뒤, 학생 제출 현황이나 성적 처리는 어디에서 확인해야 하나? --topics data_etl/processed/guide_topics.jsonl --pages data_etl/processed/pdf_pages.jsonl --top-k 4 --max-steps 4
## User Query
과제를 새로 만들고 제출 기간과 파일 제출을 설정한 뒤, 학생 제출 현황이나 성적 처리는 어디에서 확인해야 하나?

## Step 1: search_topics
reason: 과제 생성 후 제출 설정과 학생 제출 현황, 성적 처리 확인 위치가 어떤 메뉴/단계에 있는지 전체 흐름을 먼저 찾기 위해
query: 과제 새로 만들기 제출 기간 파일 제출 학생 제출 현황 성적 처리 확인 위치
S1-1. topic score=8.602 file=004_Q._과제_제출물_일괄_다운로드가_불가합니다.md topic=0
   질문: 과제 제출물 일괄 다운로드가 중간에 멈출 때 어떻게 해결하나요?
   결과: 문제가 되는 파일을 제거하거나 제출 형식을 조정한 뒤 과제 제출물을 정상적으로 확인하거나 다운로드할 수 있다.
S1-2. topic score=7.996 file=003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md topic=0
   질문: 오프라인으로 채점한 과제나 시험 점수를 new eTL에 입력하려면 어떻게 해야 하나요?
   결과: 오프라인 과제 또는 시험용 과제가 생성되고, 성적 메뉴에서 학습자별 점수와 코멘트를 입력할 수 있다.
S1-3. topic score=7.217 file=002_과제.md topic=1
   질문: 과제 제출물을 한 번에 다운로드할 수 없을 때 어떻게 하나요?
   결과: 과제

## 13. Graph feature 추출

Graph RAG까지 볼 때만 실행합니다.

API를 사용합니다.


In [22]:
LIMIT = 30
CONCURRENCY = 2

print("LIMIT:", LIMIT)
print("OUTPUT:", TOPIC_FEATURES)


LIMIT: 30
OUTPUT: data_etl/processed/topic_features.jsonl


In [23]:
run_script(
    "scripts_etl/11_extract_graph_features.py",
    "--input", TOPICS,
    "--output", TOPIC_FEATURES,
    "--limit", LIMIT,
    "--concurrency", CONCURRENCY,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\11_extract_graph_features.py --input data_etl/processed/guide_topics.jsonl --output data_etl/processed/topic_features.jsonl --limit 30 --concurrency 2
[info] topic records=12 done=0 todo=12
[ok] (1/12) 000_📑서울대학교_new_eTL_상세_가이드_(교수자용).md::0
[ok] (2/12) 001_FAQ_모음.md::0
[ok] (3/12) 000_📑서울대학교_new_eTL_상세_가이드_(교수자용).md::1
[ok] (4/12) 000_📑서울대학교_new_eTL_상세_가이드_(교수자용).md::2
[ok] (5/12) 000_📑서울대학교_new_eTL_상세_가이드_(교수자용).md::3
[ok] (6/12) 000_📑서울대학교_new_eTL_상세_가이드_(교수자용).md::4
[ok] (7/12) 002_과제.md::0
[ok] (8/12) 002_과제.md::2
[ok] (9/12) 002_과제.md::1
[ok] (10/12) 003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md::0
[ok] (11/12) 004_Q._과제_제출물_일괄_다운로드가_불가합니다.md::0
[ok] (12/12) 003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md::1
[info] output: data_etl\processed\topic_features.jsonl


## 14. Topic graph 생성

11번 결과를 graph JSON으로 만듭니다.

`ADD_SIMILARITY`가 `False`이면 API를 사용하지 않습니다. `True`이면 similarity edge 생성을 위해 embedding API를 사용합니다.


In [24]:
ADD_SIMILARITY = False

print("ADD_SIMILARITY:", ADD_SIMILARITY)
print("GRAPH:", TOPIC_GRAPH)


ADD_SIMILARITY: False
GRAPH: data_etl/indexes/topic_graph_with_similarity.json


In [25]:
args = [
    "--input", TOPIC_FEATURES,
    "--output", TOPIC_GRAPH,
]
if ADD_SIMILARITY:
    args.append("--add-similarity")

run_script("scripts_etl/12_build_topic_graph.py", *args)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\12_build_topic_graph.py --input data_etl/processed/topic_features.jsonl --output data_etl/indexes/topic_graph_with_similarity.json
nodes=168 edges=170
data_etl\indexes\topic_graph_with_similarity.json


## 15. Graph viewer HTML 생성

브라우저에서 볼 수 있는 graph HTML 파일을 만듭니다.

API를 사용하지 않습니다.


In [26]:
VIEWER_OUTPUT = "results/etl_graph_viewer_sigma.html"

print("VIEWER_OUTPUT:", VIEWER_OUTPUT)


VIEWER_OUTPUT: results/etl_graph_viewer_sigma.html


In [27]:
run_script(
    "scripts_etl/13_export_sigma_viewer.py",
    "--input", TOPIC_GRAPH,
    "--output", VIEWER_OUTPUT,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\13_export_sigma_viewer.py --input data_etl/indexes/topic_graph_with_similarity.json --output results/etl_graph_viewer_sigma.html
results\etl_graph_viewer_sigma.html
nodes=168 edges=170


## 16. Graph RAG 답변

Graph에서 관련 topic을 찾고 LLM으로 답변합니다.

API를 사용합니다.


In [28]:
QUERY = "과제 생성, 제출 설정, 성적 처리와 관련된 화면과 설정은 어떻게 연결되어 있나?"
TOP_SEEDS = 8
MAX_TOPICS = 30

print("QUERY:", QUERY)
print("TOP_SEEDS:", TOP_SEEDS)
print("MAX_TOPICS:", MAX_TOPICS)


QUERY: 과제 생성, 제출 설정, 성적 처리와 관련된 화면과 설정은 어떻게 연결되어 있나?
TOP_SEEDS: 8
MAX_TOPICS: 30


In [29]:
run_script(
    "scripts_etl/14_graph_rag_answer.py",
    QUERY,
    "--graph", TOPIC_GRAPH,
    "--top-seeds", TOP_SEEDS,
    "--max-topics", MAX_TOPICS,
)


$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_etl\14_graph_rag_answer.py 과제 생성, 제출 설정, 성적 처리와 관련된 화면과 설정은 어떻게 연결되어 있나? --graph data_etl/indexes/topic_graph_with_similarity.json --top-seeds 8 --max-topics 30
## Seed Phrase Nodes
1. score=12.84 type=Screen label=성적
2. score=11.52 type=Setting label=과제
3. score=11.52 type=Screen label=과제
4. score=6.48 type=Task label=오프라인 과제/시험용 과제 생성
5. score=5.16 type=Task label=게시판 생성
6. score=4.35 type=Setting label=제출물 유형: 오프라인 제출
7. score=4.11 type=Task label=메뉴 구성 설정
8. score=4.05 type=Screen label=성적 입력

## Graph Retrieval Summary
expanded_phrase_nodes=8
retrieved_topics=10
analysis_mode=overview
accepted_topics=0
rejected_topics=0
other_topics=10

## Representative Topics
Top graph matches:
  [C1] file=003_Q._오프라인_과제_시험_점수를_new_eTL에_입력하고_싶습니다.md topic=0 outcome=오프라인 과제 또는 시험용 과제가 생성되고, 성적 메뉴에서 학습자별 점수와 코멘트를 입력할 수 있다 score=23.67
       작업/질문: 오프라인으로 채점한 과제나 시험 점수를 new eTL에 입력하려면 어떻게 해야 하나요?
       결론: 오프라인 과제 